# 01 — Bronze: Ingest raw CSVs

Reads every raw CSV from the Volume, tags each row with source metadata
(country, division, season) from the file PATH, writes to two Delta tables
in the `bronze` schema:

- `workspace.bronze.matches_main`
- `workspace.bronze.matches_extra`

No cleaning here. Different files have different columns (46 distinct
column layouts found in exploration) — Bronze keeps everything as-is,
using Delta's schema evolution (`mergeSchema`) to handle that. Fixing the
column layouts into one consistent shape is Silver's job, not Bronze's.

In [ ]:
import os
import pandas as pd

RAW_PATH = "/Volumes/workspace/default/football_raw"

## Step 1: Discover files

In [ ]:
main_files = []
for root, dirs, files in os.walk(f"{RAW_PATH}/main_leagues"):
    for f in files:
        if f.endswith(".csv"):
            main_files.append(os.path.join(root, f))

extra_files = []
for root, dirs, files in os.walk(f"{RAW_PATH}/extra_leagues"):
    for f in files:
        if f.endswith(".csv"):
            extra_files.append(os.path.join(root, f))

print(f"Main league files: {len(main_files)}")
print(f"Extra league files: {len(extra_files)}")

## Step 2: Helper functions

`read_csv_robust` tries UTF-8 first (catches both plain UTF-8 and
UTF-8-with-BOM), falls back to cp1252 — matches what we found in
exploration: files are genuinely mixed encoding, not uniform.

In [ ]:
def read_csv_robust(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")

def parse_main_path(path):
    """.../main_leagues/{country}/{division}/{season}.csv"""
    parts = path.replace("\\", "/").split("/")
    return parts[-3], parts[-2], parts[-1].replace(".csv", "")

def parse_extra_path(path):
    """.../extra_leagues/{CODE}.csv"""
    return path.replace("\\", "/").split("/")[-1].replace(".csv", "")

## Step 3: Ingest main leagues — one file at a time, schema evolution enabled

In [ ]:
target_main = "workspace.bronze.matches_main"
ingested, failed = 0, []

for f in main_files:
    try:
        df = read_csv_robust(f)
        country, division, season = parse_main_path(f)

        df["source_country"] = country
        df["source_division"] = division
        df["source_season"] = season
        df["source_file"] = f

        # Everything as string at Bronze - avoids type conflicts between
        # files with different column layouts fighting each other on
        # write. Real typing happens in Silver.
        df = df.astype(str)

        spark_df = spark.createDataFrame(df)
        (spark_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(target_main))

        ingested += 1
        if ingested % 50 == 0:
            print(f"  ...{ingested}/{len(main_files)} ingested")

    except Exception as e:
        failed.append((f, str(e)))

print(f"\nMain leagues: {ingested} ingested, {len(failed)} failed")
for f, err in failed[:10]:
    print(f"  FAILED: {f} \u2014 {err}")

## Step 4: Ingest extra leagues — same pattern, separate table

In [ ]:
target_extra = "workspace.bronze.matches_extra"
ingested_extra, failed_extra = 0, []

for f in extra_files:
    try:
        df = read_csv_robust(f)
        df["source_code"] = parse_extra_path(f)
        df["source_file"] = f
        df = df.astype(str)

        spark_df = spark.createDataFrame(df)
        (spark_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(target_extra))

        ingested_extra += 1

    except Exception as e:
        failed_extra.append((f, str(e)))

print(f"Extra leagues: {ingested_extra} ingested, {len(failed_extra)} failed")
for f, err in failed_extra:
    print(f"  FAILED: {f} \u2014 {err}")

## Step 5: Verify

In [ ]:
main_count = spark.table(target_main).count()
extra_count = spark.table(target_extra).count()
main_cols = len(spark.table(target_main).columns)
extra_cols = len(spark.table(target_extra).columns)

print(f"workspace.bronze.matches_main:  {main_count:,} rows, {main_cols} columns")
print(f"workspace.bronze.matches_extra: {extra_count:,} rows, {extra_cols} columns")